# RAG Demo — Pipeline Walkthrough

A minimal, modular Retrieval-Augmented Generation pipeline. Each stage is a standalone module that reads from one folder under `data/` and writes to the next, so you can run, swap, or upgrade each block independently.

**Stages** (built incrementally):
1. **Document processing** — load raw files, extract to markdown ← *covered below*
2. **Chunker** — split markdown into overlapping chunks ← *covered below*
3. Embedder — turn chunks into vectors
4. Vector store — store and search vectors
5. Retriever — top-k chunks for a query
6. Answerer — Claude generates a cited answer

## Stage 1 — Document Processing

- **Input:** `data/raw/` — drop your PDFs, `.md`, or `.txt` files here.
- **Output:** `data/markdown/` — one `.md` per input file, with YAML frontmatter (`title`, `authors`, `source`).
- **Extractor:** `pymupdf` for both PDF text and metadata (see `document_processing.pdf_proc.extract_pdf_basic`); pass-through for native text/markdown.

Two entry points:
- `extract_doc(path)` — pure function, takes one file, returns a `Document`. Use it to inspect the result for a single file.
- `extract_docs(input, output)` — runs `extract_doc` over a folder and writes `.md` (with frontmatter) to disk.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from document_processing import extract_text, extract_docs
from chunker import chunk_doc, chunk_docs

In [2]:
RAW_DIR = ROOT / "data" / "raw"
MARKDOWN_DIR = ROOT / "data" / "markdown"
CHUNKS_DIR = ROOT / "data" / "chunks"
RAW_DIR.mkdir(parents=True, exist_ok=True)
MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

print("raw dir:     ", RAW_DIR)
print("markdown dir:", MARKDOWN_DIR)
print("chunks dir:  ", CHUNKS_DIR)

raw dir:      /home/amir/source/rag-demo/data/raw
markdown dir: /home/amir/source/rag-demo/data/markdown
chunks dir:   /home/amir/source/rag-demo/data/chunks


### What's in `data/raw/`?

If empty, drop a PDF (or `.md`/`.txt`) in there and re-run.

In [3]:
sorted(p.name for p in RAW_DIR.iterdir() if p.is_file())

['Large Language Models the New Scrum Masters.pdf']

### Single-file extraction (pure function)

Run `extract_doc()` on one file and inspect the resulting `Document`.

In [4]:
files = [p for p in sorted(RAW_DIR.iterdir()) if p.is_file()]
assert files, f"No files in {RAW_DIR}. Drop a PDF/.md/.txt and re-run."

sample = files[0]
doc = extract_text(file_path=sample)

print("source:  ", doc.source.name)
print("metadata:", doc.metadata)
print("length:  ", len(doc.text), "chars")
print()
print("--- first 1500 chars ---")
print(doc.text[:1500])

source:   Large Language Models the New Scrum Masters.pdf
metadata: {'title': 'Large Language Models, the New Scrum Masters', 'authors': ['Juan Couder and Omar Ochoa']}
length:   23223 chars

--- first 1500 chars ---
Papers 
RED Innovation: Using Scrum to Develop an 
Agile Department 
2024 
Large Language Models, the New Scrum Masters 
Large Language Models, the New Scrum Masters 
Juan Couder 
ortizcoj@my.erau.edu 
Omar Ochoa 
ochoao@erau.edu 
Follow this and additional works at: https://commons.erau.edu/red-papers 
Scholarly Commons Citation 
Scholarly Commons Citation 
Couder, J., & Ochoa, O. (2024). Large Language Models, the New Scrum Masters. , (). Retrieved from 
https://commons.erau.edu/red-papers/2 
This Paper is brought to you for free and open access by the RED Innovation: Using Scrum to Develop an Agile 
Department at Scholarly Commons. It has been accepted for inclusion in Papers by an authorized administrator of 
Scholarly Commons. For more information, please contact comm

### Batch run (stage runner)

`extract_docs` extracts every supported file in `data/raw/` and writes a corresponding `.md` (with YAML frontmatter) to `data/markdown/`.

In [5]:
docs = extract_docs(input_dir=RAW_DIR, 
                    output_dir=MARKDOWN_DIR)
for d in docs:
    print(f"{d.source.name:40s} -> {len(d.text):>8} chars  {d.metadata}")

Large Language Models the New Scrum Masters.pdf ->    23223 chars  {'title': 'Large Language Models, the New Scrum Masters', 'authors': ['Juan Couder and Omar Ochoa']}


### Inspect the output folder

In [6]:
for p in sorted(MARKDOWN_DIR.iterdir()):
    if p.is_file() and p.suffix == ".md":
        print(f"{p.name:40s} {p.stat().st_size:>8} bytes")

Large Language Models the New Scrum Masters.md    23424 bytes
Large Language Models the New Scrum Masters_v1.md    23865 bytes


## Stage 2 — Chunker

- **Input:** `data/markdown/` — `.md` files with YAML frontmatter from stage 1.
- **Output:** `data/chunks/` — one `.jsonl` per source doc, one chunk per line.
- **Strategy:** fixed-size character windows with overlap (default `size=2000`, `overlap=200`).

Two entry points:
- `chunk_doc(doc)` — pure function over an in-memory `Document`.
- `chunk_docs(input, output)` — reads `.md` files via frontmatter, chunks, writes `.jsonl`.

In [7]:
CHUNK_SIZE = 2000
CHUNK_OVERLAP = 200

all_chunks = [c for d in docs for c in chunk_doc(d, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)]
print(f"{len(all_chunks)} chunks across {len(docs)} docs (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})\n")

first = all_chunks[0]
print("first chunk:")
print(f"  id:       {first.id}")
print(f"  source:   {first.source}")
print(f"  index:    {first.index}")
print(f"  metadata: {first.metadata}")
print(f"  text[:300]:\n{first.text[:300]}")

13 chunks across 1 docs (size=2000, overlap=200)

first chunk:
  id:       Large Language Models the New Scrum Masters#0
  source:   Large Language Models the New Scrum Masters.pdf
  index:    0
  metadata: {'title': 'Large Language Models, the New Scrum Masters', 'authors': ['Juan Couder and Omar Ochoa']}
  text[:300]:
Papers 
RED Innovation: Using Scrum to Develop an 
Agile Department 
2024 
Large Language Models, the New Scrum Masters 
Large Language Models, the New Scrum Masters 
Juan Couder 
ortizcoj@my.erau.edu 
Omar Ochoa 
ochoao@erau.edu 
Follow this and additional works at: https://commons.erau.edu/red-pap


### Batch run

`chunk_docs` reads every `.md` in `data/markdown/` (frontmatter + body), chunks each, and writes one `.jsonl` per doc to `data/chunks/`.

In [8]:
batched = chunk_docs(input_dir=MARKDOWN_DIR, 
                     output_dir=CHUNKS_DIR, 
                     size=CHUNK_SIZE, 
                     overlap=CHUNK_OVERLAP)

print(f"wrote {len(batched)} chunks total (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})\n")

for p in sorted(CHUNKS_DIR.iterdir()):
    if p.suffix == ".jsonl":
        print(f"{p.name:40s} {p.stat().st_size:>8} bytes")

wrote 27 chunks total (size=2000, overlap=200)

Large Language Models the New Scrum Masters.jsonl    29573 bytes
Large Language Models the New Scrum Masters_v1.jsonl    29466 bytes
